# Batch pipeline — 5 YouTube links → per-video processing → ranked against the textbooks

Your existing pipeline (Stages 1–2 extraction+dedup, Stage 3 Qwen, Stage 5 expert comparison, Stage 6 whole-video score) restructured for a batch of 5 YouTube links on one topic. **All agents below are lifted unchanged from your notebooks** — only the orchestration around them is new.

1. **Step 1 — Ingest & pre-dedup the videos.** All links are downloaded with `yt-dlp`. Duplicates are removed *before any frame work*: identical YouTube IDs are dropped without downloading; re-uploads under different IDs are caught by a content fingerprint (frames sampled evenly, pHash **and** dHash medians must both agree).
2. **Stage 2 — Frames per video.** Each surviving video gets its own folder and its own `manifest.json`. Your `FrameExtractionAgent` (with the `tpad` truncated-stream fix) and `PerceptualDedupAgent` run independently per video — the dedup window never crosses videos, so cross-video overlap stays intact for the ranking to measure.
3. **Stage 3 — Your fine-tuned Qwen, per video.** The model loads **once** (official base + on-the-fly 4-bit, your LoRA on top), then your two-pass describe→structure `ContentExtractionAgent` processes each video's frames **separately**, checkpointing into that video's manifest after every frame.
4. **Stage 4 — Comparison & ranking.** Your Stage-5 machinery: books parsed once (cached), the topic scope selected **once** so every video is judged against identical sections, then each video scored separately. The final cell computes your Stage-6 whole-video score (with segment smoothing and verdict bands) per video and prints the ranking.

**Before running:** GPU runtime (T4 is enough — the base quantizes to ~7 GB). Textbook PDFs go in the books folder (cell 8).

In [1]:
# @title 1. Setup — install dependencies & check GPU { display-mode: "form" }
!pip install -q yt-dlp imagehash pymupdf pandas sentence-transformers "transformers>=4.49" accelerate qwen-vl-utils bitsandbytes peft huggingface_hub

import json, math, re, shutil, subprocess, time, hashlib
from dataclasses import dataclass, asdict, field, replace
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
import numpy as np
import pandas as pd
import imagehash
import fitz  # PyMuPDF

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB VRAM)")
else:
    print("⚠️ No GPU — Runtime → Change runtime type → GPU before Stage 3")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 21.7 MB/s eta 0:00:00
⚠️ No GPU — Runtime → Change runtime type → GPU before Stage 3


In [3]:
# @title 2. Batch configuration (new) { display-mode: "form" }

@dataclass
class BatchConfig:
    # --- the 5 links (same topic) ---
    youtube_urls: tuple = (
        "https://www.youtube.com/watch?v=RpgyCJBbl5E",   # ← EDIT all five
        "https://www.youtube.com/watch?v=M3_pLsDdeuU",
        "https://www.youtube.com/watch?v=tWVWeAqZ0WU",
        "https://www.youtube.com/watch?v=gXgEDyodOJU",
        "https://www.youtube.com/watch?v=rHemTXFJIPM",
    )
    topic: str = ""                 # ← EDIT: shared topic, e.g. "vector embeddings".
                                    #   Selects the textbook scope ONCE, so all videos are
                                    #   ranked against the same sections. Empty → prompted.
    max_height: int = 720           # download quality cap (frames are rescaled later anyway)
    cookies_file: str = ""          # path to cookies.txt if YouTube bot-checks Colab

    # --- video-level pre-dedup (step 1) ---
    fp_samples: int = 8             # frames sampled per video for the content fingerprint
    fp_hash_threshold: int = 10     # median Hamming distance (of 256 bits); BOTH pHash and
                                    #   dHash medians must be ≤ this to call it a re-upload
    fp_duration_tol_pct: float = 2.0

    # --- Stage-6 whole-video score (used for the final ranking) ---
    smooth_max_frames: int = 1
    smooth_margin: float = 0.05
    min_gap_sec: int = 60
    bands: tuple = ((0.75, "STRONG alignment with the textbooks"),
                    (0.55, "PARTIAL alignment — notable gaps"),
                    (0.35, "WEAK alignment — large portions not grounded"),
                    (0.00, "MINIMAL alignment"))

    work_dir: str = "/content/batch_pipeline"

BATCH = BatchConfig()
WORK = Path(BATCH.work_dir); WORK.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR = WORK / "videos"; VIDEOS_DIR.mkdir(exist_ok=True)
BATCH_MANIFEST_PATH = WORK / "batch_manifest.json"

def now_iso(): return datetime.now(timezone.utc).isoformat(timespec="seconds")
def save_batch(m): BATCH_MANIFEST_PATH.write_text(json.dumps(m, indent=2))
def load_batch():
    return json.loads(BATCH_MANIFEST_PATH.read_text()) if BATCH_MANIFEST_PATH.exists() else None
def video_dir(video_id): d = WORK / video_id; d.mkdir(exist_ok=True); return d
def load_vman(video_id):
    p = video_dir(video_id) / "manifest.json"
    return json.loads(p.read_text()) if p.exists() else None

print(f"{len(BATCH.youtube_urls)} links queued · topic: {BATCH.topic or '(will prompt in Stage 4)'}")


5 links queued · topic: (will prompt in Stage 4)


In [4]:
# @title 3. Your PipelineConfig (unchanged — per-video work dirs are derived from it)
@dataclass
class PipelineConfig:
    # --- frame extraction ---
    frame_interval_sec: int = 120          # 1 frame every 2 minutes
    image_format: str = "jpg"              # jpg keeps size down; use "png" for lossless
    jpeg_quality: int = 2                  # ffmpeg -q:v scale: 2 = high quality, 31 = worst
    accurate_seek: bool = False            # True = one ffmpeg seek per frame (exact timestamps, slower)
                                           # False = single-pass fps filter (fast, timestamps ≈ n*interval)
    scale_max_width: int = 1280            # downscale huge videos; 1280px is plenty for slide OCR.
                                           # set to 0 to keep original resolution

    # --- perceptual dedup ---
    hash_size: int = 16                    # 16 → 256-bit hashes (finer than default 8)
    phash_threshold: int = 12              # Hamming distance below which frames count as duplicates
    dhash_threshold: int = 12              #   (out of hash_size^2 = 256 bits; ~5% of bits)
    require_both: bool = True              # True: BOTH pHash and dHash must agree it's a dupe
                                           #   (conservative — fewer false deletions)
    compare_window: int = 3                # compare against the last N *kept* frames, not just 1

    # --- paths ---
    work_dir: str = "/content/pipeline"

    def __post_init__(self):
        self.frames_dir     = str(Path(self.work_dir) / "frames_raw")
        self.kept_dir       = str(Path(self.work_dir) / "frames_kept")
        self.duplicates_dir = str(Path(self.work_dir) / "frames_duplicates")
        self.manifest_path  = str(Path(self.work_dir) / "manifest.json")

CFG_P = PipelineConfig()          # defaults; each video gets its own copy via
                                  # replace(CFG_P, work_dir=<video folder>) — __post_init__
                                  # re-derives frames_raw/kept/duplicates/manifest per video
print(json.dumps({k: v for k, v in asdict(CFG_P).items() if not k.endswith("_dir")
                  and k != "manifest_path"}, indent=2))


{
  "frame_interval_sec": 120,
  "image_format": "jpg",
  "jpeg_quality": 2,
  "accurate_seek": false,
  "scale_max_width": 1280,
  "hash_size": 16,
  "phash_threshold": 12,
  "dhash_threshold": 12,
  "require_both": true,
  "compare_window": 3
}


## Step 1 — Ingest the 5 links & pre-dedup the videos

Replaces the single-video upload cells. Two passes, both before any frame is extracted: exact YouTube-ID repeats are skipped without downloading; then a content fingerprint (evenly-sampled frames, pHash + dHash, **both** medians must be ≤ `fp_hash_threshold`) catches re-uploads under different IDs. Duration must also agree within `fp_duration_tol_pct`%. Dropped videos keep an auditable entry in `batch_manifest.json` with `duplicate_of` and the measured distances — nothing is deleted.

In [5]:
# @title 4. Step 1 — download, fingerprint, pre-dedup { display-mode: "form" }

YT_ID_RE = re.compile(
    r"(?:youtu\.be/|youtube\.com/(?:watch\?(?:.*&)?v=|shorts/|embed/|live/))([A-Za-z0-9_-]{11})")

def canonical_id(url: str):
    m = YT_ID_RE.search(url)
    return m.group(1) if m else None

def yt_download(url: str, out_dir: Path) -> dict:
    import yt_dlp
    opts = {
        "format": f"bv*[height<={BATCH.max_height}][ext=mp4]+ba[ext=m4a]/"
                  f"b[height<={BATCH.max_height}][ext=mp4]/b",
        "outtmpl": str(out_dir / "%(id)s.%(ext)s"),
        "merge_output_format": "mp4",
        "quiet": True, "no_warnings": True, "noplaylist": True,
    }
    if BATCH.cookies_file:
        opts["cookiefile"] = BATCH.cookies_file
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
    vid = info["id"]
    path = next(out_dir.glob(f"{vid}.*"))
    return {"video_id": vid, "title": info.get("title", vid),
            "duration_sec": float(info.get("duration") or 0),
            "channel": info.get("channel", ""), "url": url, "path": str(path)}

def fingerprint(path: str, duration: float, n: int):
    """(pHash list, dHash list) of n frames sampled evenly across the video."""
    tmp = Path(path).parent / (Path(path).stem + "_fp")
    shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
    phashes, dhashes = [], []
    for i in range(n):
        t = duration * (i + 0.5) / n
        f = tmp / f"{i}.jpg"
        subprocess.run(["ffmpeg", "-y", "-ss", f"{t:.2f}", "-i", path, "-frames:v", "1",
                        "-vf", "scale=320:-2", "-q:v", "4", str(f)], capture_output=True)
        if f.exists():
            img = Image.open(f)
            phashes.append(imagehash.phash(img, hash_size=16))
            dhashes.append(imagehash.dhash(img, hash_size=16))
    shutil.rmtree(tmp, ignore_errors=True)
    return phashes, dhashes

def median_distance(h1, h2):
    n = min(len(h1), len(h2))
    if n == 0: return 999
    d = sorted(h1[i] - h2[i] for i in range(n))
    return d[n // 2]

# ---- ID pass ----
records, seen_ids = [], {}
for url in BATCH.youtube_urls:
    vid = canonical_id(url)
    if vid and vid in seen_ids:
        records.append({"url": url, "video_id": vid, "status": "duplicate_id",
                        "duplicate_of": vid})
        print(f"✂  {url} → same ID as an earlier link ({vid}), skipping download")
        continue
    if vid: seen_ids[vid] = url
    records.append({"url": url, "video_id": vid, "status": "pending"})

# ---- download pass ----
for rec in records:
    if rec["status"] != "pending": continue
    try:
        print("⬇  downloading", rec["url"], "…")
        info = yt_download(rec["url"], VIDEOS_DIR)
        rec.update(info); rec["status"] = "downloaded"
        print(f'   ✓ {info["title"]} — {rec["duration_sec"]/60:.1f} min')
    except Exception as e:
        rec["status"], rec["error"] = "download_failed", str(e)
        print(f"   ✗ failed: {e}")

# ---- content pass ----
downloaded = [r for r in records if r["status"] == "downloaded"]
for r in downloaded:
    r["_fp"] = fingerprint(r["path"], r["duration_sec"], BATCH.fp_samples)

for i, a in enumerate(downloaded):
    if a["status"] != "downloaded": continue
    for b in downloaded[i + 1:]:
        if b["status"] != "downloaded": continue
        if a["duration_sec"] and abs(a["duration_sec"] - b["duration_sec"]) \
                > BATCH.fp_duration_tol_pct / 100 * max(a["duration_sec"], b["duration_sec"]):
            continue
        pd_ = median_distance(a["_fp"][0], b["_fp"][0])
        dd_ = median_distance(a["_fp"][1], b["_fp"][1])
        if pd_ <= BATCH.fp_hash_threshold and dd_ <= BATCH.fp_hash_threshold:
            b["status"], b["duplicate_of"] = "duplicate_content", a["video_id"]
            b["fp_distance"] = {"phash": int(pd_), "dhash": int(dd_)}
            print(f'✂  "{b["title"]}" is a re-upload of "{a["title"]}" '
                  f"(median distances phash {pd_}, dhash {dd_} ≤ {BATCH.fp_hash_threshold})")

for r in records: r.pop("_fp", None)
kept = [r for r in records if r["status"] == "downloaded"]
for r in kept: r["status"] = "kept"

save_batch({"created_utc": now_iso(), "topic": BATCH.topic,
            "batch_config": asdict(BATCH), "videos": records})
print(f"\n✅ Step 1 done → {len(kept)} unique video(s) of {len(BATCH.youtube_urls)} links:")
for r in records:
    tag = {"kept": "✓", "duplicate_id": "✂ id-dup", "duplicate_content": "✂ content-dup",
           "download_failed": "✗ failed"}.get(r["status"], "?")
    print(f'  {tag:14s} {r.get("video_id","?")}  {r.get("title", r["url"])}')


⬇  downloading https://www.youtube.com/watch?v=RpgyCJBbl5E …
   ✓ Introduction to Graphs | Data Structure & Algorithms — 26.1 min
⬇  downloading https://www.youtube.com/watch?v=M3_pLsDdeuU …
   ✓ G-1. Introduction to Graph | Types | Different Conventions Used — 13.7 min
⬇  downloading https://www.youtube.com/watch?v=tWVWeAqZ0WU …
   ✓ Graph Algorithms for Technical Interviews - Full Course — 132.3 min
⬇  downloading https://www.youtube.com/watch?v=gXgEDyodOJU …
   ✓ Data structures: Introduction to graphs — 16.7 min
⬇  downloading https://www.youtube.com/watch?v=rHemTXFJIPM …
   ✓ Graph - Data Structures in Python #8 — 28.8 min

✅ Step 1 done → 5 unique video(s) of 5 links:
  ✓              RpgyCJBbl5E  Introduction to Graphs | Data Structure & Algorithms
  ✓              M3_pLsDdeuU  G-1. Introduction to Graph | Types | Different Conventions Used
  ✓              tWVWeAqZ0WU  Graph Algorithms for Technical Interviews - Full Course
  ✓              gXgEDyodOJU  Data structures: Introdu

## Stage 2 — Frame extraction per video (your agents, unchanged)

`probe_video`, `FrameExtractionAgent` (including the `tpad=stop_mode=clone` fix for video streams that end before the container does, and the expected-frame-count check) and `PerceptualDedupAgent` are copied verbatim from your Stage 1–2 notebook. The only new code is the loop: each video gets its own `PipelineConfig` via `replace(CFG_P, work_dir=<video folder>)`, so `frames_raw/`, `frames_kept/`, `frames_duplicates/` and `manifest.json` are derived per video by your own `__post_init__` — extraction and dedup never mix frames from different videos.

In [6]:
# @title 5. probe_video — verbatim from your Stage 1–2 notebook
def probe_video(path: str) -> dict:
    """Return container + first-video-stream metadata via ffprobe."""
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height,avg_frame_rate,codec_name",
        "-show_entries", "format=duration,size,format_name",
        "-of", "json", path,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffprobe failed:\n{r.stderr}")
    meta = json.loads(r.stdout)
    stream, fmt = meta["streams"][0], meta["format"]
    num, den = (stream.get("avg_frame_rate") or "0/1").split("/")
    fps = (float(num) / float(den)) if float(den) else 0.0
    return {
        "path": path,
        "codec": stream.get("codec_name"),
        "width": stream.get("width"),
        "height": stream.get("height"),
        "fps": round(fps, 3),
        "duration_sec": float(fmt["duration"]),
        "size_mb": round(int(fmt["size"]) / 1e6, 1),
        "container": fmt.get("format_name"),
    }


In [7]:
# @title 6. FrameExtractionAgent — verbatim
class FrameExtractionAgent:
    """Extracts 1 frame every `frame_interval_sec` seconds and builds a timestamped manifest."""

    def __init__(self, cfg: PipelineConfig, video_info: dict):
        self.cfg, self.info = cfg, video_info

    # ---------- public ----------
    def run(self) -> dict:
        out_dir = Path(self.cfg.frames_dir)
        for old in out_dir.glob(f"*.{self.cfg.image_format}"):
            old.unlink()                                # idempotent re-runs

        t0 = time.time()
        if self.cfg.accurate_seek:
            frames = self._extract_accurate(out_dir)
        else:
            frames = self._extract_fast(out_dir)
        elapsed = time.time() - t0

        manifest = {
            "video": self.info,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "config": asdict(self.cfg),
            "stages": {"extraction": {"mode": "accurate" if self.cfg.accurate_seek else "fast",
                                      "elapsed_sec": round(elapsed, 1),
                                      "frame_count": len(frames)}},
            "frames": frames,
        }
        Path(self.cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
        print(f"✅ Extracted {len(frames)} frames in {elapsed:.1f}s → {out_dir}")
        return manifest

    # ---------- internals ----------
    def _scale_filter(self) -> str:
        w = self.cfg.scale_max_width
        return f",scale='min({w},iw)':-2" if w else ""

    def _frame_record(self, path: Path, index: int, ts: float) -> dict:
        return {
            "frame_id": path.stem,
            "index": index,
            "timestamp_sec": round(ts, 2),
            "timestamp_hms": time.strftime("%H:%M:%S", time.gmtime(ts)),
            "path": str(path),
            "status": "extracted",        # → kept / duplicate after dedup
            "phash": None, "dhash": None,
            "dedup": None,
        }

    def _extract_fast(self, out_dir: Path) -> list:
        pattern = str(out_dir / f"frame_%04d.{self.cfg.image_format}")
        dur = self.info["duration_sec"]
        # tpad clones the last real frame out to the container duration: screen recordings
        # often stop emitting video frames before the audio/container ends, which would
        # otherwise silently truncate sampling (e.g. 7 frames from a 14.5-min video).
        vf = (f"tpad=stop_mode=clone:stop_duration={dur},"
              f"fps=1/{self.cfg.frame_interval_sec}{self._scale_filter()}")
        cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
               "-i", self.info["path"], "-vf", vf,
               "-t", f"{dur + self.cfg.frame_interval_sec / 2:.3f}",
               "-q:v", str(self.cfg.jpeg_quality),
               "-vsync", "vfr",   # NOT -fps_mode: Colab ships ffmpeg 4.x, which lacks it
               pattern]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg failed:\n{r.stderr}")
        frames = []
        for i, p in enumerate(sorted(out_dir.glob(f"frame_*.{self.cfg.image_format}"))):
            frames.append(self._frame_record(p, i, i * self.cfg.frame_interval_sec))
        return frames

    def _extract_accurate(self, out_dir: Path) -> list:
        frames, ts, i = [], 0.0, 0
        while ts < self.info["duration_sec"]:
            p = out_dir / f"frame_{i:04d}.{self.cfg.image_format}"
            vf = f"scale='min({self.cfg.scale_max_width},iw)':-2" if self.cfg.scale_max_width else "null"
            cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                   "-ss", f"{ts:.3f}", "-i", self.info["path"],
                   "-frames:v", "1", "-vf", vf,
                   "-q:v", str(self.cfg.jpeg_quality), str(p)]
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError(f"ffmpeg failed at t={ts}:\n{r.stderr}")
            if not p.exists() and frames:
                # timestamp is past the last real video frame — clone the previous one
                shutil.copy(frames[-1]["path"], p)
            if p.exists():
                frames.append(self._frame_record(p, i, ts))
            ts += self.cfg.frame_interval_sec
            i += 1
            if i % 10 == 0:
                print(f"  … {i} frames ({ts/60:.0f} min in)")
        return frames


In [8]:
# @title 7. PerceptualDedupAgent — verbatim
class PerceptualDedupAgent:
    """Drops visually near-identical frames using pHash + dHash Hamming distance
    against a sliding window of recently *kept* frames."""

    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg

    def run(self, manifest: dict) -> dict:
        cfg = self.cfg
        kept_window = []          # [(phash, dhash, frame_id), …] most recent last
        n_kept = n_dup = 0
        t0 = time.time()

        for rec in manifest["frames"]:
            img = Image.open(rec["path"])
            ph = imagehash.phash(img, hash_size=cfg.hash_size)
            dh = imagehash.dhash(img, hash_size=cfg.hash_size)
            rec["phash"], rec["dhash"] = str(ph), str(dh)

            match = self._find_match(ph, dh, kept_window)
            if match:
                rec["status"] = "duplicate"
                rec["dedup"] = match
                dst = Path(cfg.duplicates_dir) / Path(rec["path"]).name
                shutil.move(rec["path"], dst)
                rec["path"] = str(dst)
                n_dup += 1
            else:
                rec["status"] = "kept"
                dst = Path(cfg.kept_dir) / Path(rec["path"]).name
                shutil.move(rec["path"], dst)
                rec["path"] = str(dst)
                kept_window.append((ph, dh, rec["frame_id"]))
                kept_window = kept_window[-cfg.compare_window:]
                n_kept += 1

        manifest["stages"]["perceptual_dedup"] = {
            "kept": n_kept, "duplicates": n_dup,
            "reduction_pct": round(100 * n_dup / max(1, n_kept + n_dup), 1),
            "phash_threshold": cfg.phash_threshold,
            "dhash_threshold": cfg.dhash_threshold,
            "require_both": cfg.require_both,
            "elapsed_sec": round(time.time() - t0, 1),
        }
        Path(cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
        print(f"✅ Kept {n_kept}, moved {n_dup} duplicates "
              f"({manifest['stages']['perceptual_dedup']['reduction_pct']}% reduction)")
        return manifest

    def _find_match(self, ph, dh, window):
        """Return match info if (ph, dh) is a near-duplicate of anything in the window."""
        for prev_ph, prev_dh, prev_id in reversed(window):
            p_dist, d_dist = ph - prev_ph, dh - prev_dh
            p_hit = p_dist <= self.cfg.phash_threshold
            d_hit = d_dist <= self.cfg.dhash_threshold
            is_dup = (p_hit and d_hit) if self.cfg.require_both else (p_hit or d_hit)
            if is_dup:
                return {"duplicate_of": prev_id,
                        "phash_distance": int(p_dist),
                        "dhash_distance": int(d_dist)}
        return None


# ---------------- per-video driver (new) ----------------
batch = load_batch()
assert batch, "Run Step 1 first."
for vrec in [v for v in batch["videos"] if v["status"] == "kept"]:
    vid = vrec["video_id"]
    print(f'\n▶ {vid} — {vrec["title"]}')
    vcfg = replace(CFG_P, work_dir=str(video_dir(vid)))
    for dd in (vcfg.frames_dir, vcfg.kept_dir, vcfg.duplicates_dir):
        Path(dd).mkdir(parents=True, exist_ok=True)

    info = probe_video(vrec["path"])
    expected = math.floor(info["duration_sec"] / vcfg.frame_interval_sec) + 1

    manifest = FrameExtractionAgent(vcfg, info).run()
    got = len(manifest["frames"])
    if got < expected:
        print(f'  ⚠️ expected ~{expected} frames, got {got} '
              f'(video stream may end before the container — tpad should prevent this)')
    manifest = PerceptualDedupAgent(vcfg).run(manifest)

    manifest["video_id"], manifest["title"], manifest["url"] = vid, vrec["title"], vrec["url"]
    Path(vcfg.manifest_path).write_text(json.dumps(manifest, indent=2))
print("\n✅ Stage 2 done — one manifest.json per video under", WORK)



▶ RpgyCJBbl5E — Introduction to Graphs | Data Structure & Algorithms
✅ Extracted 14 frames in 91.8s → /content/batch_pipeline/RpgyCJBbl5E/frames_raw
✅ Kept 14, moved 0 duplicates (0.0% reduction)

▶ M3_pLsDdeuU — G-1. Introduction to Graph | Types | Different Conventions Used
✅ Extracted 7 frames in 125.9s → /content/batch_pipeline/M3_pLsDdeuU/frames_raw
✅ Kept 7, moved 0 duplicates (0.0% reduction)

▶ tWVWeAqZ0WU — Graph Algorithms for Technical Interviews - Full Course
✅ Extracted 67 frames in 314.6s → /content/batch_pipeline/tWVWeAqZ0WU/frames_raw
✅ Kept 66, moved 1 duplicates (1.5% reduction)

▶ gXgEDyodOJU — Data structures: Introduction to graphs
✅ Extracted 9 frames in 15.8s → /content/batch_pipeline/gXgEDyodOJU/frames_raw
✅ Kept 9, moved 0 duplicates (0.0% reduction)

▶ rHemTXFJIPM — Graph - Data Structures in Python #8
✅ Extracted 15 frames in 123.5s → /content/batch_pipeline/rHemTXFJIPM/frames_raw
✅ Kept 15, moved 0 duplicates (0.0% reduction)

✅ Stage 2 done — one manifest.

## Stage 3 — Your fine-tuned Qwen, each video's frames passed separately

Config, model load (official `Qwen/Qwen2.5-VL-7B-Instruct` base + on-the-fly 4-bit quantization, your LoRA adapter attached — exactly the setup you converged on) and the two-pass describe→structure `ContentExtractionAgent` (with the heuristic-structuring fallback) are verbatim from your Stage 3 notebook. The model loads **once**; the new driver loop then runs the agent on each video in turn with a per-video config clone, so every frame result is checkpointed into *that video's* manifest — video B never starts until video A's manifest is fully written, and a crash resumes mid-video thanks to your existing resume logic in `run()`.

In [9]:
# @title 8. Your Stage3Config — verbatim (work_dir is overridden per video)
@dataclass
class Stage3Config:
    # --- your model ---
    model_id: str = "shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm"
    is_lora_adapter: bool = True        # this repo is an Unsloth-trained LoRA adapter
    base_model_id: str = "Qwen/Qwen2.5-VL-7B-Instruct"
                                        # official base + on-the-fly 4-bit quantization (below).
                                        # The unsloth pre-quantized repo the adapter was trained from
                                        # ("unsloth/qwen2.5-vl-7b-instruct-unsloth-bnb-4bit") breaks on
                                        # current transformers — the LoRA attaches identically to this one.
    run_mode: str = "local"             # "local" (load into Colab GPU) or "endpoint" (HF Inference Endpoint)
    endpoint_url: str = ""              # only for run_mode="endpoint", e.g. "https://xxxx.endpoints.huggingface.cloud"

    # --- generation ---
    max_new_tokens: int = 1024
    temperature: float = 0.1            # near-deterministic: we want faithful transcription, not creativity
    load_in_4bit: bool = True           # quantize the official base on load → ~7 GB VRAM, fits a T4
    max_retries: int = 3

    # --- frames input ---
    frame_interval_sec: int = 30        # ONLY used to reconstruct timestamps when manifest.json is absent
    image_format: str = "jpg"

    # --- paths ---
    work_dir: str = "/content/stage3"

    def __post_init__(self):
        self.frames_dir    = str(Path(self.work_dir) / "frames_kept")
        self.manifest_path = str(Path(self.work_dir) / "manifest.json")

CFG3 = Stage3Config()
print(json.dumps({k: v for k, v in asdict(CFG3).items()
                  if k not in ("work_dir", "frames_dir", "manifest_path")}, indent=2))


{
  "model_id": "shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm",
  "is_lora_adapter": true,
  "base_model_id": "Qwen/Qwen2.5-VL-7B-Instruct",
  "run_mode": "local",
  "endpoint_url": "",
  "max_new_tokens": 1024,
  "temperature": 0.1,
  "load_in_4bit": true,
  "max_retries": 3,
  "frame_interval_sec": 30,
  "image_format": "jpg"
}


In [10]:
# @title 9. Load model + processor ONCE — verbatim
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

model, processor = None, None
if CFG3.run_mode == "local":
    # T4 GPUs don't support bfloat16 — pick compute dtype accordingly
    compute_dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

    load_kwargs = {"device_map": "auto"}
    if "bnb-4bit" in CFG3.base_model_id or "bnb-4bit" in CFG3.model_id:
        # pre-quantized repo: pass NOTHING extra, let its baked-in quantization config drive.
        # NOTE: unsloth's dynamically-quantized repos break on some transformers versions —
        # if this path errors, use the official base + load_in_4bit=True instead (the default).
        print("Base repo is pre-quantized 4-bit — deferring to its baked-in config.")
    elif CFG3.load_in_4bit:
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True)
        print(f"Quantizing to 4-bit on load (compute dtype: {compute_dtype})")
    else:
        load_kwargs["torch_dtype"] = "auto"

    load_id = CFG3.base_model_id if CFG3.is_lora_adapter else CFG3.model_id
    t0 = time.time()
    model = AutoModelForImageTextToText.from_pretrained(load_id, **load_kwargs)

    if CFG3.is_lora_adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, CFG3.model_id)
        print("Attached LoRA adapter:", CFG3.model_id)
        print("Active adapters:", model.active_adapters)

    try:
        processor = AutoProcessor.from_pretrained(CFG3.model_id)
    except Exception:
        processor = AutoProcessor.from_pretrained(CFG3.base_model_id)

    model.eval()
    print(f"✅ Model ready in {time.time()-t0:.0f}s")
else:
    print("run_mode='endpoint' — model stays on HF infrastructure, nothing to load here.")


Quantizing to 4-bit on load (compute dtype: torch.float16)


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

chat_template.jinja:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

✅ Model ready in 879s


In [11]:
# @title 10. ContentExtractionAgent — verbatim
DESCRIBE_PROMPT = """Describe this frame from a technical video course in detail.
Transcribe every piece of visible text exactly as shown: the title, all bullet points, labels, and captions.
If code is visible, reproduce it verbatim, preserving formatting.
If there is a figure, chart, or diagram, describe what it depicts and how it is structured."""

STRUCTURE_PROMPT = """Convert the following description of a video frame into a JSON object with exactly these fields:
{{"slide_title": string or null, "content_text": string, "code": string or null, "diagram_description": string or null, "frame_type": "slide" | "code_editor" | "terminal" | "diagram" | "talking_head" | "other"}}
Rules: put transcribed body text in content_text; put verbatim code in code; put figure/diagram explanation in diagram_description; use null when absent.
Respond with ONLY the JSON object.

Description:
{description}"""


def _lenient_json(text: str):
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", t, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
    return None


def _structure_heuristic(description: str) -> dict:
    """Model-free fallback: structure a raw description with simple rules."""
    text = description.strip()
    code = None
    fences = re.findall(r"```[a-zA-Z]*\n?(.*?)```", text, re.DOTALL)
    if fences:
        code = "\n\n".join(f.strip() for f in fences)
        text = re.sub(r"```[a-zA-Z]*\n?.*?```", "", text, flags=re.DOTALL).strip()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    title = lines[0] if lines and len(lines[0]) < 80 else None
    low = description.lower()
    if code or "code editor" in low or "ide" in low:
        ftype = "code_editor"
    elif any(k in low for k in ("terminal", "command line", "shell prompt")):
        ftype = "terminal"
    elif any(k in low for k in ("chart", "graph", "diagram", "figure", "plot", "axis")):
        ftype = "diagram"
    elif any(k in low for k in ("person", "presenter", "speaker", "face", "webcam")):
        ftype = "talking_head"
    else:
        ftype = "slide"
    return {"slide_title": title, "content_text": text, "code": code,
            "diagram_description": text if ftype == "diagram" else None,
            "frame_type": ftype, "_structured_by": "heuristic"}


class ContentExtractionAgent:
    def __init__(self, cfg, model=None, processor=None):
        self.cfg, self.model, self.processor = cfg, model, processor
        if cfg.run_mode == "endpoint":
            from huggingface_hub import InferenceClient
            self.client = InferenceClient(base_url=cfg.endpoint_url)

    # ---------- one frame: describe → structure ----------
    def extract(self, image_path: str) -> dict:
        last_err = None
        for attempt in range(1, self.cfg.max_retries + 1):
            try:
                description = self._describe(image_path)          # pass 1 (vision)
                structured = self._structure(description)          # pass 2 (text-only)
                if structured is None:
                    structured = _structure_heuristic(description) # fallback (no model)
                structured["raw_description"] = description        # never lose the original
                return structured
            except Exception as e:
                last_err = e
                wait = 2 ** attempt
                print(f"  retry {attempt}/{self.cfg.max_retries} after error: {e} (waiting {wait}s)")
                time.sleep(wait)
        return {"slide_title": None, "content_text": None, "code": None,
                "diagram_description": None, "frame_type": "error",
                "raw_description": None, "_error": str(last_err)}

    def _describe(self, image_path: str) -> str:
        content = [{"type": "image", "image": image_path},
                   {"type": "text", "text": DESCRIBE_PROMPT}]
        return self._chat(content, max_new_tokens=self.cfg.max_new_tokens)

    def _structure(self, description: str):
        prompt = STRUCTURE_PROMPT.format(description=description)
        raw = self._chat([{"type": "text", "text": prompt}], max_new_tokens=self.cfg.max_new_tokens)
        parsed = _lenient_json(raw)
        if parsed is not None and isinstance(parsed, dict):
            parsed.setdefault("frame_type", "other")
            parsed["_structured_by"] = "model"
            return parsed
        return None

    # ---------- shared chat call, image optional ----------
    def _chat(self, content: list, max_new_tokens: int) -> str:
        if self.cfg.run_mode == "endpoint":
            return self._chat_endpoint(content, max_new_tokens)
        from qwen_vl_utils import process_vision_info
        messages = [{"role": "user", "content": content}]
        text = self.processor.apply_chat_template(messages, tokenize=False,
                                                  add_generation_prompt=True)
        has_image = any(c.get("type") == "image" for c in content)
        image_inputs = process_vision_info(messages)[0] if has_image else None
        inputs = self.processor(text=[text], images=image_inputs,
                                padding=True, return_tensors="pt").to(self.model.device)
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens,
                                      do_sample=self.cfg.temperature > 0,
                                      temperature=max(self.cfg.temperature, 1e-5))
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
        return self.processor.batch_decode(trimmed, skip_special_tokens=True,
                                           clean_up_tokenization_spaces=False)[0]

    def _chat_endpoint(self, content: list, max_new_tokens: int) -> str:
        import base64
        parts = []
        for c in content:
            if c["type"] == "image":
                b64 = base64.b64encode(Path(c["image"]).read_bytes()).decode()
                parts.append({"type": "image_url",
                              "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})
            else:
                parts.append({"type": "text", "text": c["text"]})
        resp = self.client.chat.completions.create(
            model="tgi", messages=[{"role": "user", "content": parts}],
            max_tokens=max_new_tokens, temperature=self.cfg.temperature)
        return resp.choices[0].message.content

    # ---------- whole manifest, checkpointed + resumable ----------
    def run(self, manifest: dict) -> dict:
        todo = [f for f in manifest["frames"]
                if f.get("status") in (None, "kept") and "extracted_content" not in f]
        done = len([f for f in manifest["frames"] if "extracted_content" in f])
        print(f"{len(todo)} frames to process ({done} already done — resuming)")

        t0 = time.time()
        for i, rec in enumerate(todo, 1):
            t = time.time()
            rec["extracted_content"] = self.extract(rec["path"])
            rec["extraction_meta"] = {"model": self.cfg.model_id,
                                      "mode": self.cfg.run_mode,
                                      "structured_by": rec["extracted_content"].get("_structured_by", "error"),
                                      "elapsed_sec": round(time.time() - t, 1)}
            Path(self.cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
            title = rec["extracted_content"].get("slide_title") or "—"
            print(f'[{i}/{len(todo)}] {rec["frame_id"]} @ {rec["timestamp_hms"]} '
                  f'({rec["extraction_meta"]["elapsed_sec"]}s, '
                  f'{rec["extraction_meta"]["structured_by"]}) → {str(title)[:60]}')

        manifest.setdefault("stages", {})["content_extraction"] = {
            "model": self.cfg.model_id, "mode": self.cfg.run_mode,
            "frames_processed": len(todo),
            "elapsed_sec": round(time.time() - t0, 1)}
        Path(self.cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
        print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min")
        return manifest


# ---------------- per-video driver (new) ----------------
batch = load_batch()
for vrec in [v for v in batch["videos"] if v["status"] == "kept"]:
    vid = vrec["video_id"]
    vcfg3 = replace(CFG3, work_dir=str(video_dir(vid)))   # __post_init__ → this video's
                                                          # frames_kept/ + manifest.json
    manifest = load_vman(vid)
    assert manifest, f"No manifest for {vid} — run Stage 2 first."
    print(f'\n▶ {vid} — {manifest.get("title", vid)}')
    agent = ContentExtractionAgent(vcfg3, model, processor)
    agent.run(manifest)          # checkpoints to this video's manifest after every frame
print("\n✅ Stage 3 done — extracted_content written per frame, per video.")



▶ RpgyCJBbl5E — Introduction to Graphs | Data Structure & Algorithms
14 frames to process (0 already done — resuming)


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


KeyboardInterrupt: 

## Stage 4 — Comparison with the textbooks in the folder, then the ranking

Your Stage-5 machinery, run once where it can be shared and per-video where it must be separate:

- **Shared** (fair-ranking guarantee): textbook parsing (TOC-first, font-fallback, cached — verbatim), the embedding model, and the topic scope. The scope is selected once from `BATCH.topic` using your keyword + context-floor logic, with your `SELECTED_OVERRIDE` escape hatch intact — so all videos are scored against **identical** textbook chunks.
- **Per video**: your frame-text assembly, the collapse-into-groups pass, the groups×chunks similarity matrix, and groundedness / topic-coverage / headline scores — computed independently per manifest, written into it as `comparison`, with a per-video `stage5_report.json` + CSV in each video folder.

The final cell applies your Stage-6 segment smoothing and whole-video score (time-coverage × topic-coverage harmonic mean, verdict bands) to each video and prints the ranked table.

In [ ]:
# @title 11. Your Stage5Config — verbatim (video_title replaced by the shared BATCH.topic)
@dataclass
class Stage5Config:
    video_title: str = ""               # ← EDIT: e.g. "Working with Vector Embeddings".
                                        #   Leave "" to be prompted from the manifest/interactively.

    # --- embeddings ---
    embedding_model: str = "BAAI/bge-small-en-v1.5"   # solid default; "all-MiniLM-L6-v2" is faster

    # --- textbook chunking ---
    chunk_words: int = 350              # ~450 tokens
    chunk_overlap_words: int = 60

    # --- topic scoping ---
    auto_scope: bool = True             # True: auto-select EVERY section (across ALL books) whose title
                                        #   contains a keyword from the video title, with context checked
    scope_min_context: float = 0.45     # semantic floor for keyword hits — filters false friends
                                        #   ("graphics" won't ride in on "graph")
    scope_candidates: int = 15          # candidates shown for the manual path / override
    min_section_words: int = 100        # ignore trivially short TOC entries (title pages etc.)

    # --- scoring ---
    similarity_threshold: float = 0.75  # your "75%" — calibrate with step 4 before trusting it
    collapse_threshold: float = 0.95    # consecutive frames above this cosine are grouped
                                        # (compensates for the skipped duplicate-eliminator stage)
    exclude_frame_types: tuple = ("talking_head", "error")

    # --- paths ---
    work_dir: str = "/content/stage5"

    def __post_init__(self):
        self.books_dir  = str(Path(self.work_dir) / "books")
        self.cache_dir  = str(Path(self.work_dir) / "cache")
        self.report_dir = str(Path(self.work_dir) / "report")

CFG5 = Stage5Config(work_dir=str(WORK))       # books/, cache/, report/ shared by the batch
for dd in (CFG5.books_dir, CFG5.cache_dir, CFG5.report_dir):
    Path(dd).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(CFG5), indent=2))


In [ ]:
# @title 12. Textbooks — put your PDF folder in place { display-mode: "form" }
# Option A — already on disk: set books_dir below.  Option B — copy from Drive (uncomment):
# from google.colab import drive
# drive.mount("/content/drive")
# BOOKS_SRC = "/content/drive/MyDrive/pipeline/textbooks"     # ← EDIT
# for p in Path(BOOKS_SRC).glob("*.pdf"):
#     shutil.copy(p, Path(CFG5.books_dir) / p.name)


In [ ]:
# @title 13. Textbook parser (TOC-first, font-size fallback, cached) — verbatim, runs ONCE
def _file_key(path: Path) -> str:
    return hashlib.md5(f"{path.name}:{path.stat().st_size}".encode()).hexdigest()[:12]

def _chunk_words(text: str, size: int, overlap: int):
    words = text.split()
    step = max(1, size - overlap)
    for start in range(0, max(1, len(words)), step):
        piece = words[start:start + size]
        if len(piece) < 30 and start > 0:   # tail too small to stand alone
            break
        yield " ".join(piece)

def _sections_from_toc(doc) -> list:
    toc = doc.get_toc()          # [[level, title, page], ...] 1-based pages
    if not toc:
        return []
    secs = []
    for i, (level, title, page) in enumerate(toc):
        end_page = doc.page_count
        for lvl2, _, pg2 in toc[i + 1:]:
            if lvl2 <= level:
                end_page = pg2 - 1
                break
        secs.append({"level": level, "title": title.strip(),
                     "page_start": page, "page_end": max(page, end_page)})
    return secs

def _sections_from_fonts(doc) -> list:
    """No bookmarks: treat lines with font size ≥ 1.3× the body size as headings."""
    sizes = {}
    lines = []          # (page_1based, size, text)
    for pno in range(doc.page_count):
        for block in doc.load_page(pno).get_text("dict")["blocks"]:
            for line in block.get("lines", []):
                text = "".join(s["text"] for s in line.get("spans", [])).strip()
                if not text:
                    continue
                size = round(max(s["size"] for s in line["spans"]), 1)
                sizes[size] = sizes.get(size, 0) + len(text)
                lines.append((pno + 1, size, text))
    if not lines:
        return []
    body = max(sizes, key=sizes.get)                     # most common size = body text
    heads = [(pg, sz, tx) for pg, sz, tx in lines
             if sz >= body * 1.3 and 3 < len(tx) < 120]
    secs = []
    for i, (pg, sz, tx) in enumerate(heads):
        end = heads[i + 1][0] if i + 1 < len(heads) else doc.page_count
        secs.append({"level": 1 if sz >= body * 1.6 else 2, "title": tx,
                     "page_start": pg, "page_end": max(pg, end)})
    return secs

def parse_book(pdf_path: Path, cfg) -> dict:
    cache = Path(cfg.cache_dir) / f"{_file_key(pdf_path)}.json"
    if cache.exists():
        book = json.loads(cache.read_text())
        print(f'  {pdf_path.name}: cached ({len(book["sections"])} sections)')
        return book

    doc = fitz.open(pdf_path)
    secs = _sections_from_toc(doc)
    source = "toc"
    if not secs:
        secs = _sections_from_fonts(doc)
        source = "font-heuristic"

    page_text = [doc.load_page(p).get_text("text") for p in range(doc.page_count)]
    kept = []
    for s in secs:
        text = "\n".join(page_text[s["page_start"] - 1:s["page_end"]])
        if len(text.split()) < cfg.min_section_words:
            continue
        s["chunks"] = [{"text": ch,
                        "pages": f'{s["page_start"]}–{s["page_end"]}'}
                       for ch in _chunk_words(text, cfg.chunk_words, cfg.chunk_overlap_words)]
        if s["chunks"]:
            kept.append(s)
    doc.close()

    book = {"file": pdf_path.name, "structure_source": source, "sections": kept}
    cache.write_text(json.dumps(book))
    print(f'  {pdf_path.name}: parsed via {source} → {len(kept)} usable sections, '
          f'{sum(len(s["chunks"]) for s in kept)} chunks')
    return book


BOOKS = [parse_book(p, CFG5) for p in sorted(Path(CFG5.books_dir).glob("*.pdf"))]
assert BOOKS, f"No PDFs found in {CFG5.books_dir}"

# flat list of (book_idx, section_idx) for scoping
ALL_SECTIONS = [(bi, si) for bi, b in enumerate(BOOKS) for si in range(len(b["sections"]))]
print(f"\n{len(BOOKS)} books · {len(ALL_SECTIONS)} sections total")


In [ ]:
# @title 14. Embedding model — verbatim, loaded ONCE for scope + all videos
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(CFG5.embedding_model)
print("Embedding model:", CFG5.embedding_model,
      "· dim:", embedder.get_sentence_embedding_dimension())

def embed(texts: list) -> np.ndarray:
    """L2-normalized embeddings → dot product == cosine similarity."""
    return embedder.encode(texts, normalize_embeddings=True,
                           show_progress_bar=len(texts) > 50, batch_size=64)


In [ ]:
# @title 15. Build the scope ONCE from the shared topic — your keyword + context auto-selection

if not BATCH.topic:
    BATCH.topic = input("Topic shared by the videos (selects the textbook scope): ").strip()
    _b = load_batch(); _b["topic"] = BATCH.topic; save_batch(_b)

STOPWORDS = {"a","an","the","and","or","of","in","on","for","with","to","is","are",
             "how","what","why","using","introduction","tutorial","course","part",
             "chapter","lesson","video","complete","guide","beginners"}

def title_keywords(title: str) -> list:
    words = re.findall(r"[a-zA-Z]{3,}", title.lower())
    return [w for w in words if w not in STOPWORDS]

def keyword_in(kw: str, text: str) -> bool:
    """Exact word match with simple plural handling — no prefix matching."""
    for tok in re.findall(r"[a-zA-Z]+", text.lower()):
        if tok == kw or tok == kw + "s" or tok == kw + "es" or kw == tok + "s":
            return True
    return False

def section_label(bi, si):
    b, s = BOOKS[bi], BOOKS[bi]["sections"][si]
    return f'{s["title"]} — {b["file"]} (pp. {s["page_start"]}–{s["page_end"]}, level {s["level"]})'

KEYWORDS = title_keywords(BATCH.topic)
print(f'Topic: "{BATCH.topic}" → keywords: {KEYWORDS}\n')

# semantic score for every section (used as context check AND for the manual/override list)
section_title_texts = [f'{Path(BOOKS[bi]["file"]).stem}: {BOOKS[bi]["sections"][si]["title"]}'
                       for bi, si in ALL_SECTIONS]
title_vec = embed([BATCH.topic])[0]
sem_scores = embed(section_title_texts) @ title_vec

# ---- pass 1+2: keyword hit AND context floor ----
hits = []
for idx, (bi, si) in enumerate(ALL_SECTIONS):
    s = BOOKS[bi]["sections"][si]
    if any(keyword_in(kw, s["title"]) for kw in KEYWORDS) \
            and sem_scores[idx] >= CFG5.scope_min_context:
        hits.append((bi, si))

# ---- pass 3: drop sections nested inside an already-selected section of the same book ----
def _contains(a, b):   # a contains b (same book)
    return a["page_start"] <= b["page_start"] and a["page_end"] >= b["page_end"] \
           and (a["page_start"], a["page_end"]) != (b["page_start"], b["page_end"])

AUTO_SELECTED = []
for bi, si in hits:
    s = BOOKS[bi]["sections"][si]
    if any(b2 == bi and _contains(BOOKS[b2]["sections"][s2], s) for b2, s2 in hits):
        continue        # a matching parent already covers these pages
    AUTO_SELECTED.append((bi, si))

if CFG5.auto_scope and AUTO_SELECTED:
    per_book = {}
    for bi, si in AUTO_SELECTED:
        per_book.setdefault(BOOKS[bi]["file"], []).append(si)
    print(f"Auto-selected {len(AUTO_SELECTED)} sections across {len(per_book)} book(s):")
    for bi, si in AUTO_SELECTED:
        idx = ALL_SECTIONS.index((bi, si))
        print(f"  ✓ {sem_scores[idx]:.3f}  {section_label(bi, si)}")
    books_no_hit = [b["file"] for b in BOOKS if b["file"] not in per_book]
    if books_no_hit:
        print("\nNo qualifying section in:", ", ".join(books_no_hit),
              "\n(no title keyword match above the context floor — lower scope_min_context "
              "or use SELECTED_OVERRIDE if that's wrong)")
else:
    if CFG5.auto_scope:
        print("⚠️ No section title matched the keywords — falling back to semantic ranking. "
              "Pick indices with SELECTED_OVERRIDE in the next cell.")
    order = np.argsort(-sem_scores)[:CFG5.scope_candidates]
    print("\nRanked candidates:")
    for rank, idx in enumerate(order):
        bi, si = ALL_SECTIONS[idx]
        print(f"  [{rank}] {sem_scores[idx]:.3f}  {section_label(bi, si)}")
    CANDIDATES = [ALL_SECTIONS[i] for i in order]


In [ ]:
# @title 16. Finalize the scope — your override cell, unchanged
SELECTED_OVERRIDE = None    # e.g. [0, 2] to hand-pick from the ranked candidate list instead

if SELECTED_OVERRIDE is not None:
    scope = [CANDIDATES[i] for i in SELECTED_OVERRIDE]
elif CFG5.auto_scope and AUTO_SELECTED:
    scope = AUTO_SELECTED
else:
    raise ValueError("No auto-selection available — set SELECTED_OVERRIDE from the candidate list above.")

scope_chunks, scope_meta = [], []
for bi, si in scope:
    s = BOOKS[bi]["sections"][si]
    for ci, ch in enumerate(s["chunks"]):
        scope_chunks.append(ch["text"])
        scope_meta.append({"book": BOOKS[bi]["file"], "section": s["title"],
                           "pages": ch["pages"], "chunk_id": f"{bi}.{si}.{ci}"})

print("Scoring scope:")
for bi, si in scope:
    print("  •", section_label(bi, si))
print(f"{len(scope_chunks)} textbook chunks in scope "
      f"from {len(set(m['book'] for m in scope_meta))} book(s)")


In [ ]:
# @title 17. Score each video separately against the shared scope { display-mode: "form" }

def frame_text(rec: dict) -> str:                      # verbatim from your Stage 5
    c = rec.get("extracted_content", {}) or {}
    parts = [c.get("slide_title"), c.get("content_text"),
             c.get("code"), c.get("diagram_description"), c.get("raw_description")]
    return "\n".join(str(p) for p in parts if p)

def scoreable_frames(manifest: dict) -> list:          # your Stage-5 cell 3, as a function
    out = []
    for rec in manifest["frames"]:
        if "extracted_content" not in rec:
            continue
        ftype = rec["extracted_content"].get("frame_type", "other")
        txt = frame_text(rec)
        if ftype in CFG5.exclude_frame_types or not txt.strip():
            continue
        out.append({"frame_id": rec["frame_id"],
                    "timestamp_hms": rec.get("timestamp_hms", "?"),
                    "timestamp_sec": rec.get("timestamp_sec", 0),
                    "frame_type": ftype, "text": txt})
    return out

chunk_vecs = embed(scope_chunks)      # once — the scope is shared

def score_video(frames: list):        # your Stage-5 scoring cell, as a function
    frame_vecs = embed([f["text"] for f in frames])
    groups, current = [], [0]
    for i in range(1, len(frames)):
        if float(frame_vecs[i] @ frame_vecs[current[-1]]) >= CFG5.collapse_threshold:
            current.append(i)
        else:
            groups.append(current); current = [i]
    groups.append(current)
    group_vecs = np.stack([frame_vecs[g].mean(axis=0) for g in groups])
    group_vecs /= np.linalg.norm(group_vecs, axis=1, keepdims=True)

    S = group_vecs @ chunk_vecs.T
    best_chunk, best_score = S.argmax(axis=1), S.max(axis=1)
    thr = CFG5.similarity_threshold
    video_groundedness = float((best_score >= thr).mean())
    topic_coverage = float((S.max(axis=0) >= thr).mean())
    headline = (0.0 if video_groundedness + topic_coverage == 0 else
                2 * video_groundedness * topic_coverage /
                (video_groundedness + topic_coverage))
    rows = []
    for gi, g in enumerate(groups):
        for fi in g:
            f, m = frames[fi], scope_meta[best_chunk[gi]]
            rows.append({"frame_id": f["frame_id"], "time": f["timestamp_hms"],
                         "t_sec": f["timestamp_sec"], "type": f["frame_type"], "group": gi,
                         "score": round(float(best_score[gi]), 3),
                         "matched": bool(best_score[gi] >= thr),
                         "best_section": m["section"], "pages": m["pages"], "book": m["book"]})
    results = {"video_groundedness": round(video_groundedness, 4),
               "topic_coverage": round(topic_coverage, 4),
               "headline_score": round(headline, 4),
               "frames_scored": len(frames), "content_groups": len(groups),
               "chunks_in_scope": len(scope_chunks)}
    return results, sorted(rows, key=lambda r: r["t_sec"])

batch = load_batch()
for vrec in [v for v in batch["videos"] if v["status"] == "kept"]:
    vid = vrec["video_id"]; manifest = load_vman(vid)
    frames = scoreable_frames(manifest)
    print(f'\n▶ {manifest.get("title", vid)}: {len(manifest["frames"])} frames → '
          f'{len(frames)} scoreable (excluded: {CFG5.exclude_frame_types})')
    if not frames:
        manifest["comparison"] = {"error": "no scoreable frames"}
        Path(video_dir(vid), "manifest.json").write_text(json.dumps(manifest, indent=2))
        continue
    results, rows = score_video(frames)
    manifest["comparison"] = {
        "topic": BATCH.topic, "created_utc": now_iso(),
        "embedding_model": CFG5.embedding_model,
        "scope": [section_label(bi, si) for bi, si in scope],
        "threshold": CFG5.similarity_threshold,
        "results": results, "frames": rows}
    Path(video_dir(vid), "manifest.json").write_text(json.dumps(manifest, indent=2))
    # per-video Stage-5-style report, same shape your Stage 6 expects
    rep = {"video_title": manifest.get("title", vid), **manifest["comparison"]}
    Path(video_dir(vid), "stage5_report.json").write_text(json.dumps(rep, indent=2))
    pd.DataFrame(rows).to_csv(Path(video_dir(vid), "stage5_frames.csv"), index=False)
    print(f'  groundedness {results["video_groundedness"]:.1%} · '
          f'coverage {results["topic_coverage"]:.1%} · '
          f'headline {results["headline_score"]:.1%}')


In [ ]:
# @title 18. Ranking — your Stage-6 whole-video score per video, then the table { display-mode: "form" }

def build_segments(rows, thr, cfg):        # verbatim from your Stage 6 (cfg = BATCH here)
    matched = rows["matched"].to_list()
    scores  = rows["score"].to_list()
    sm = matched[:]
    i = 0
    while i < len(sm):
        if not sm[i]:
            j = i
            while j < len(sm) and not sm[j]:
                j += 1
            run = j - i
            shallow = all(scores[k] >= thr - cfg.smooth_margin for k in range(i, j))
            inside = i > 0 and j < len(sm)
            if run <= cfg.smooth_max_frames and shallow and inside:
                for k in range(i, j):
                    sm[k] = True
            i = j
        else:
            i += 1
    rows["matched_smoothed"] = sm
    segs, start = [], 0
    interval = CFG_P.frame_interval_sec
    for i in range(1, len(sm) + 1):
        if i == len(sm) or sm[i] != sm[start]:
            block = rows.iloc[start:i]
            t0 = int(block["t_sec"].iloc[0])
            t1 = int(block["t_sec"].iloc[-1]) + interval
            seg = {"kind": "matched" if sm[start] else "gap",
                   "start_sec": t0, "end_sec": t1, "start": block["time"].iloc[0],
                   "duration_sec": t1 - t0, "n_frames": len(block),
                   "avg_score": round(float(block["score"].mean()), 3),
                   "sections": sorted(set(zip(block["best_section"],
                                              block["pages"], block["book"])))}
            if seg["kind"] == "gap":
                seg["minor"] = seg["duration_sec"] < cfg.min_gap_sec
                seg["shortfall"] = round(thr - seg["avg_score"], 3)
            segs.append(seg)
            start = i
    return segs

def whole_video_score(rows, results, thr):   # your Stage-6 score cell, as a function
    segments = build_segments(rows, thr, BATCH)
    interval = CFG_P.frame_interval_sec
    dur_matched = sum(s["duration_sec"] for s in segments if s["kind"] == "matched")
    dur_gap     = sum(s["duration_sec"] for s in segments if s["kind"] == "gap")
    time_coverage = dur_matched / max(1, dur_matched + dur_gap)
    tc = results["topic_coverage"]
    final = (0.0 if time_coverage + tc == 0
             else 2 * time_coverage * tc / (time_coverage + tc))
    verdict = next(label for cut, label in BATCH.bands if final >= cut)
    return final, verdict, time_coverage, segments

batch = load_batch()
ranked = []
for vrec in [v for v in batch["videos"] if v["status"] == "kept"]:
    vid = vrec["video_id"]; manifest = load_vman(vid)
    comp = manifest.get("comparison", {})
    if "results" not in comp:
        continue
    rows = pd.DataFrame(comp["frames"])
    final, verdict, time_cov, segments = whole_video_score(
        rows, comp["results"], comp["threshold"])
    manifest["whole_video"] = {"final_score": round(final, 4), "verdict": verdict,
                               "time_coverage": round(time_cov, 4),
                               "segments": segments}
    Path(video_dir(vid), "manifest.json").write_text(json.dumps(manifest, indent=2))
    ranked.append({"video_id": vid, "title": manifest.get("title", vid),
                   "url": vrec["url"], "final_score": round(final, 4),
                   "verdict": verdict, "time_coverage": round(time_cov, 4),
                   **comp["results"]})

ranked.sort(key=lambda r: -r["final_score"])
print(f'RANKING — topic "{batch["topic"]}", threshold {CFG5.similarity_threshold}, '
      f'{len(scope_chunks)} chunks in scope\n')
print(f'{"#":<3}{"final":<8}{"time":<8}{"topic":<8}{"grndd":<8}{"headline":<10}title')
for i, r in enumerate(ranked, 1):
    print(f'{i:<3}{r["final_score"]:<8.3f}{r["time_coverage"]:<8.3f}'
          f'{r["topic_coverage"]:<8.3f}{r["video_groundedness"]:<8.3f}'
          f'{r["headline_score"]:<10.3f}{r["title"][:48]}')
    print(f'   └ {r["verdict"]}')

dropped = [v for v in batch["videos"] if v["status"].startswith("duplicate")]
if dropped:
    print("\nDropped in Step 1 pre-dedup:")
    for v in dropped:
        print(f'  ✂ {v.get("title", v["url"])} → duplicate of {v["duplicate_of"]}')

batch["ranking"] = ranked
save_batch(batch)
Path(WORK, "batch_report.json").write_text(json.dumps(
    {"topic": batch["topic"], "created_utc": now_iso(),
     "scope": [section_label(bi, si) for bi, si in scope],
     "threshold": CFG5.similarity_threshold,
     "ranking": ranked, "videos": batch["videos"]}, indent=2))

export = WORK / "_export"; shutil.rmtree(export, ignore_errors=True); export.mkdir()
shutil.copy(WORK / "batch_report.json", export / "batch_report.json")
shutil.copy(BATCH_MANIFEST_PATH, export / "batch_manifest.json")
for vrec in batch["videos"]:
    if vrec["status"] == "kept":
        for name in ("manifest.json", "stage5_report.json", "stage5_frames.csv"):
            p = video_dir(vrec["video_id"]) / name
            if p.exists():
                shutil.copy(p, export / f'{vrec["video_id"]}_{name}')
zip_path = shutil.make_archive(str(WORK / "batch_results"), "zip", export)
print("\nSaved:", WORK / "batch_report.json")
print("Zipped:", zip_path)
# from google.colab import files; files.download(zip_path)
